# CVD Synthesis Optimization via Bayesian Optimization
### Full Pipeline Walkthrough
---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
print('Setup complete')

## 1. Data Generation (simulates literature mining)

In [ ]:
from data_preprocessing import generate_raw_cvd_data
raw_df = generate_raw_cvd_data(n_samples=150, random_state=42)
print(f'Raw dataset: {raw_df.shape}')
raw_df.head()

In [ ]:
print('Missing values per column:')
print(raw_df.isnull().sum())
print('\nBasic stats:')
raw_df.describe()

## 2. Preprocessing Pipeline
Steps: unit standardization → outlier removal → imputation → categorical encoding → scaling

In [ ]:
from data_preprocessing import preprocess_pipeline
proc_df, scaler, t_scalers = preprocess_pipeline(raw_df)
print(f'\nProcessed shape: {proc_df.shape}')
proc_df.head(3)

## 3. Feature Engineering
Physics-informed derived features + composite quality score

In [ ]:
from feature_engineering import add_engineered_features, build_quality_score, get_feature_columns
proc_df = add_engineered_features(proc_df)
proc_df = build_quality_score(proc_df)
feat_cols = get_feature_columns(proc_df)
print(f'Feature columns ({len(feat_cols)}):')
print(feat_cols)
print(f'\nQuality score stats:')
print(proc_df['quality_score'].describe())

In [ ]:
from visualization import plot_quality_distribution
X = proc_df[feat_cols].values
y = proc_df['quality_score'].values
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(y, bins=25, color='#2563EB', alpha=0.7, edgecolor='white')
ax.axvline(y.mean(), color='red', ls='--', lw=1.5, label=f'Mean={y.mean():.3f}')
ax.axvline(y.max(),  color='green', ls='--', lw=1.5, label=f'Max={y.max():.3f}')
ax.set(xlabel='Quality Score', ylabel='Count', title='Quality Score Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Gaussian Process Regression

In [ ]:
from gpr_model import CVDGaussianProcess
from evaluation import train_test_evaluation, calibration_analysis, kfold_evaluation
from sklearn.metrics import r2_score

gp = CVDGaussianProcess(kernel_type='matern', n_restarts=5)
metrics, y_test, y_pred, y_std, X_train, X_test, y_train = train_test_evaluation(gp, X, y, test_size=0.2)
r2 = r2_score(y_test, y_pred)
print(f'\nTest metrics: {metrics}')
print(f'R² = {r2:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
lo = min(y_test.min(), y_pred.min()) - 0.05
hi = max(y_test.max(), y_pred.max()) + 0.05
ax.errorbar(y_test, y_pred, yerr=y_std, fmt='o', alpha=0.6, color='#2563EB',
            ecolor='#94A3B8', elinewidth=1, capsize=3, ms=5, label='Pred ± 1σ')
ax.plot([lo,hi],[lo,hi],'k--',lw=1.5,label='Perfect fit')
ax.text(0.05,0.92,f'R²={r2:.3f}',transform=ax.transAxes,bbox=dict(fc='white',alpha=0.7))
ax.set(xlabel='Actual Quality Score',ylabel='Predicted Quality Score',title='Predicted vs Actual')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
calib_df = calibration_analysis(y_test, y_pred, y_std)
calib_df

In [ ]:
cv_df, cv_agg = kfold_evaluation(CVDGaussianProcess, X, y, k=5)
print('\nCV Aggregate:')
cv_agg

## 5. Feature Sensitivity Analysis

In [ ]:
from sensitivity_analysis import sensitivity_report, partial_dependence_all
gp_full = CVDGaussianProcess(kernel_type='matern', n_restarts=5).build()
gp_full.fit(X_train, y_train, feature_names=feat_cols)
imp_df = sensitivity_report(gp_full, feat_cols, X_test, y_test)
imp_df

In [ ]:
df_imp = imp_df.sort_values('Importance_Norm', ascending=True).tail(12)
fig, ax = plt.subplots(figsize=(8, max(4, len(df_imp)*0.4)))
bars = ax.barh(df_imp['Feature'], df_imp['Importance_Norm'], color='#2563EB', alpha=0.85)
ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=9)
ax.set(xlabel='Normalised Importance', title='Feature Importance (Permutation-based)')
plt.tight_layout()
plt.show()

In [ ]:
temp_idx  = feat_cols.index('temperature')  if 'temperature'  in feat_cols else 0
press_idx = feat_cols.index('log_pressure') if 'log_pressure' in feat_cols else 1

n_grid = 30
temps   = np.linspace(-2.5, 2.5, n_grid)
log_prs = np.linspace(-2.5, 2.5, n_grid)
T_grid, P_grid = np.meshgrid(temps, log_prs)
Z = np.zeros_like(T_grid)

for i in range(n_grid):
    for j in range(n_grid):
        X_mod = X_test.mean(axis=0, keepdims=True).copy()
        X_mod[0, temp_idx]  = T_grid[i, j]
        X_mod[0, press_idx] = P_grid[i, j]
        mu, _ = gp_full.predict(X_mod, return_std=True)
        Z[i, j] = float(mu[0])

fig, ax = plt.subplots(figsize=(8,6))
cf = ax.contourf(T_grid, P_grid, Z, levels=20, cmap='viridis')
fig.colorbar(cf, ax=ax, label='Predicted Quality Score')
ax.set(xlabel='Temperature (scaled)', ylabel='Log Pressure (scaled)',
       title='Predicted Quality: Temperature × Log Pressure')
plt.tight_layout()
plt.show()

## 6. Bayesian Optimization
EI acquisition, 35 iterations, 12 seed observations

In [ ]:
from bayesian_optimization import BayesianOptimizer

ref_gp = CVDGaussianProcess(kernel_type='matern', n_restarts=5).build()
ref_gp.fit(X, y, feature_names=feat_cols)

def simulated_experiment(x):
    mu, _ = ref_gp.predict(x, return_std=True)
    return float(np.clip(mu[0] + np.random.normal(0, 0.02), 0, 1))

param_bounds = {f: (-2.5, 2.5) for f in feat_cols}
rng_seed = np.random.default_rng(0)
n_seed = 12
X_seed = np.column_stack([rng_seed.uniform(-2.5, 2.5, n_seed) for _ in feat_cols])
y_seed = np.array([simulated_experiment(x.reshape(1,-1)) for x in X_seed])

optimizer = BayesianOptimizer(
    gp_class=CVDGaussianProcess,
    objective_fn=simulated_experiment,
    param_bounds=param_bounds,
    acq_func='EI',
    n_candidates=2000,
    random_state=42,
)
optimizer.initialize(X_seed, y_seed)
bo_results = optimizer.run(n_iterations=35, verbose=True)

In [ ]:
hist = bo_results['history']
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(hist['iteration'], hist['y_next'], alpha=0.5, s=30, color='#2563EB', label='Observed quality')
ax.plot(hist['iteration'], hist['best_so_far'], color='#2563EB', lw=2.5, label='Best so far')
ax.axvline(x=n_seed, color='grey', ls='--', lw=1.5, label=f'Seed ({n_seed} obs)')
ax.set(xlabel='Iteration', ylabel='Quality Score', title='Bayesian Optimization Convergence')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Baseline Comparison

In [ ]:
from baseline_comparison import run_comparison
n_trials = int(bo_results['history'].shape[0])
summary_df, histories, rs_res, gs_res = run_comparison(
    bo_results, simulated_experiment, param_bounds,
    n_trials=n_trials, random_state=42
)
summary_df

In [ ]:
color_map = {'Bayesian Optimization':'#2563EB','Random Search':'#DC2626','Grid Search':'#16A34A'}
fig, ax = plt.subplots(figsize=(10,5))
for name, hist in histories.items():
    col = color_map.get(name, 'grey')
    ax.plot(hist['trial'].values, hist['best'].values, label=name, color=col, lw=2.2)
ax.set(xlabel='Number of Experiments', ylabel='Best Quality Score',
       title='Bayesian Optimization vs Baselines')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Final Summary

In [ ]:
print('='*55)
print('  RESULTS SUMMARY')
print('='*55)
print(f'  GPR R²             : {r2:.4f}')
print(f'  GPR RMSE           : {metrics["RMSE"]}')
print(f'  GPR MAE            : {metrics["MAE"]}')
print(f'  5-Fold CV R² mean  : {cv_agg.loc["mean","R2"]}')
print(f'  BO best quality    : {bo_results["best_quality"]:.4f}')
print(f'  BO best iteration  : {bo_results["best_iteration"]}')
print(f'  Random best quality: {rs_res["best_quality"]:.4f}')
print(f'  Grid best quality  : {gs_res["best_quality"]:.4f}')
print('='*55)